
# E1 Module B, shard 2 of 2: multiplicity

This notebook runs Module B from P3-T2. The full factorial contains 600
cells: K(3) x correction(5) x rho(5) x family(2) x theta(2) x design(2).
Shard 2 runs exactly 300 cells, uses 2,000 outer repetitions, and
uses B=999 as the bootstrap budget recorded with every row. The scalar
P2-T3 harness receives the correction-specific planning level from
`tisca.multiplicity.planning_alpha`, including the Romano-Wolf schedule.
Results are checkpointed to Drive one cell at a time and can be resumed.

The empirical family requires the real M x 2 loss matrix. The setup cell
finds it on Drive or invokes the standard Colab upload dialog.


In [ ]:

import itertools
import os
import pathlib
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_URL = "https://github.com/hugogobato/Test-Informed-Simulation-Count-Algorithm-TISCA.git"
LOCAL_REPO = "/content/Test-Informed-Simulation-Count-Algorithm-TISCA"
CLONED_REPO = "/content/TISCA_repo"

if os.path.isdir(os.path.join(LOCAL_REPO, "tisca", "python")):
    SOURCE_ROOT = os.path.join(LOCAL_REPO, "tisca", "python")
elif os.path.isdir(os.path.join(CLONED_REPO, "tisca", "python")):
    SOURCE_ROOT = os.path.join(CLONED_REPO, "tisca", "python")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONED_REPO], check=True)
    SOURCE_ROOT = os.path.join(CLONED_REPO, "tisca", "python")

assert os.path.isdir(SOURCE_ROOT), f"TISCA Python package not found at {SOURCE_ROOT}"
sys.path.insert(0, SOURCE_ROOT)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("(Not on Colab / Drive mount skipped):", exc)

if os.path.isdir("/content/drive/MyDrive"):
    DRIVE_ROOT = "/content/drive/MyDrive/TISCA_E1"
else:
    DRIVE_ROOT = "/content/TISCA_E1"
    print("[WARN] Drive is not mounted; checkpointing to", DRIVE_ROOT)
os.makedirs(DRIVE_ROOT, exist_ok=True)

from tisca.outermc import engine, summarize_ocs
from tisca import multiplicity

ALPHA = 0.05
DELTA = 0.5
JMAX = 1000
MATRIX_CANDIDATES = [
    os.path.join(DRIVE_ROOT, "E1_empirical_loss_matrix.npy"),
    os.path.join(DRIVE_ROOT, "E1_empirical_loss_matrix.csv"),
    "/content/E1_empirical_loss_matrix.npy",
    "/content/E1_empirical_loss_matrix.csv",
]


def load_empirical_matrix():
    """Find or upload the real M x 2 loss matrix used by family (g)."""
    path = next((p for p in MATRIX_CANDIDATES if os.path.exists(p)), None)
    if path is None:
        try:
            from google.colab import files
            print("Upload E1_empirical_loss_matrix.npy or .csv (M x 2) when prompted.")
            uploaded = files.upload()
            if uploaded:
                name = next(iter(uploaded))
                path = os.path.join("/content", name)
        except Exception as exc:
            print("(Upload skipped):", exc)
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(
            "The empirical family needs the real M x 2 loss matrix. "
            "Place E1_empirical_loss_matrix.npy/.csv in the Drive folder or upload it."
        )
    if path.lower().endswith(".npy"):
        matrix = np.load(path)
    else:
        matrix = pd.read_csv(path, header=None).to_numpy(dtype=float)
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] != 2 or matrix.shape[0] < 2:
        raise ValueError(f"empirical matrix must have shape (M, 2), got {matrix.shape}")
    if not np.all(np.isfinite(matrix)):
        raise ValueError("empirical matrix contains non-finite values")
    print("[PASS] empirical matrix:", matrix.shape, "from", path)
    return matrix


EMPIRICAL_MATRIX = load_empirical_matrix()


def _append(row, path):
    frame = pd.DataFrame([row])
    header = not os.path.exists(path) or os.path.getsize(path) == 0
    frame.to_csv(path, mode="a", header=header, index=False)


def run_grid(grid, output_name):
    """Run and checkpoint a deterministic grid, resuming completed cell IDs."""
    output_file = os.path.join(DRIVE_ROOT, output_name)
    error_file = output_file.replace("_results.csv", "_errors.csv")
    done = set()
    if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
        old = pd.read_csv(output_file)
        if "cell_id" not in old.columns:
            raise ValueError(f"existing checkpoint lacks cell_id: {output_file}")
        done = set(old["cell_id"].astype(str))

    pending = [c for c in grid if c["cell_id"] not in done]
    print(f"{output_name}: {len(done)} completed, {len(pending)} pending, {len(grid)} expected")
    failures = []
    started = time.time()
    for cell in tqdm(pending, desc=output_name, unit="cell"):
        try:
            cfg = dict(cell["config"])
            summary, _, _ = engine.run_e1(cfg)
            row = summarize_ocs([summary]).iloc[0].to_dict()
            row.update(cell["factors"])
            row.update(
                cell_id=cell["cell_id"],
                module=cell["module"],
                projected_R=cell["config"]["R"],
                bootstrap_B=cell["config"].get("B", np.nan),
            )
            _append(row, output_file)
        except Exception as exc:
            print("[FAIL]", cell["cell_id"], repr(exc))
            failures.append({"cell_id": cell["cell_id"], "error": repr(exc)})

    if failures:
        pd.DataFrame(failures).to_csv(error_file, index=False)
        raise RuntimeError(f"{len(failures)} cells failed; see {error_file}")

    result = pd.read_csv(output_file)
    if set(result["cell_id"].astype(str)) != {c["cell_id"] for c in grid}:
        missing = sorted({c["cell_id"] for c in grid} - set(result["cell_id"].astype(str)))
        raise RuntimeError(f"checkpoint incomplete; missing {len(missing)} cells, first={missing[:3]}")
    if result["cell_id"].duplicated().any():
        raise RuntimeError("duplicate cell_id detected in checkpoint")
    print(f"[PASS] {len(result)} rows in {output_file}; elapsed {time.time() - started:.0f}s")
    return output_file, result


def download_fallback(output_file):
    try:
        from google.colab import files
        files.download(output_file)
        print("Downloaded:", output_file)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)


In [ ]:

def make_module_b_grid():
    cells = []
    index = 0
    for K, correction, rho, family, theta, design in itertools.product(
        [1, 3, 6],
        ["none", "bonferroni", "holm", "bh", "romano_wolf"],
        [-0.3, 0.0, 0.3, 0.6, 0.9],
        ["normal", "empirical"],
        [0.0, DELTA],
        ["D3", "D4"],
    ):
        alpha_plan, alpha_note = multiplicity.planning_alpha(
            correction, K, alpha=ALPHA, r=1
        )
        factors = dict(module_cell="B", family=family, rho=rho,
                       theta_mult=round(theta / DELTA, 3), design=design,
                       J0=50, B=999, K=K, correction=correction,
                       alpha_plan=alpha_plan, alpha_note=alpha_note)
        config = {
            "design": design, "family": family, "rho": rho,
            "sigma_a": 1.0, "sigma_b": 1.0, "theta": theta,
            "sigma_D": None, "R": 2000, "J0": 50, "Jmax": JMAX,
            "alpha": ALPHA, "alpha_adj": alpha_plan, "mode": 1,
            "delta": DELTA, "power_target": 0.80, "gamma": 0.20,
            "correction": correction, "K": K,
            "matrix": EMPIRICAL_MATRIX if family == "empirical" else None,
            "seed": 400000 + index, "fixed_J": None, "mcse": 0.05,
            "batch": 50, "B": 999,
        }
        cells.append(dict(cell_id=f"B_{index:04d}", module="B",
                          factors=factors, config=config))
        index += 1
    assert len(cells) == 600, len(cells)
    return cells


MODULE_B_GRID = make_module_b_grid()
print("Module B total cells:", len(MODULE_B_GRID))


In [ ]:

SHARD = 2
START, END = 300, 600
SHARD_GRID = MODULE_B_GRID[START:END]
assert len(SHARD_GRID) == 300
assert SHARD_GRID[0]["cell_id"] == f"B_{START:04d}"
assert SHARD_GRID[-1]["cell_id"] == f"B_{END - 1:04d}"
print("Module B shard", SHARD, "cells:", len(SHARD_GRID))


In [ ]:

OUTPUT_FILE, RESULTS = run_grid(SHARD_GRID, "E1_modB_shard2_results.csv")
assert len(RESULTS) == 300
assert RESULTS["cell_id"].is_unique
print("[PASS] Module B shard 2 complete")
download_fallback(OUTPUT_FILE)
